# Homework 8

Curly vs straight hair, small CNN in PyTorch on CPU. Data is the fixed `data.zip` from the
course (sha256 checked), unzipped into `data/` and not committed.

In [1]:
import json
import random
import statistics

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_num_threads(2)  # same as reference_train.py

device = torch.device("cpu")
torch.__version__

'2.9.0+cpu'

In [2]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    normalize,
])

train_dataset = datasets.ImageFolder("data/train", transform=evaluation_transform)
evaluation_dataset = datasets.ImageFolder("data/test", transform=evaluation_transform)
print(len(train_dataset), len(evaluation_dataset), train_dataset.class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True, num_workers=0,
                          generator=torch.Generator().manual_seed(SEED))
evaluation_loader = DataLoader(evaluation_dataset, batch_size=20, shuffle=False, num_workers=0)

800 201 {'curly': 0, 'straight': 1}


In [3]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)


model = HairModel().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

## Q1 - loss

**Q1: `nn.BCEWithLogitsLoss()`.** The model gives one raw logit per image, and this loss does the sigmoid and binary cross entropy in one stable step.

## Q2 - number of parameters

In [4]:
sum(p.numel() for p in model.parameters())

20073473

**Q2: 20073473.** Almost all of it is the dense layer: 32 * 99 * 99 * 64 + 64 = 20,072,512.

## Training

In [5]:
def run_epoch(model, loader, train=True):
    model.train() if train else model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.float().unsqueeze(1).to(device)
            if train:
                optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * x.size(0)
            correct += ((torch.sigmoid(logits) >= 0.5) == y).sum().item()
            total += x.size(0)
    return loss_sum / total, correct / total


def train(model, loader, epochs=10):
    h = {"train_loss": [], "train_accuracy": [], "evaluation_loss": [], "evaluation_accuracy": []}
    for epoch in range(epochs):
        tl, ta = run_epoch(model, loader, train=True)
        el, ea = run_epoch(model, evaluation_loader, train=False)
        h["train_loss"].append(tl)
        h["train_accuracy"].append(ta)
        h["evaluation_loss"].append(el)
        h["evaluation_accuracy"].append(ea)
        print(f"{epoch + 1:2d}  loss {tl:.4f}  acc {ta:.4f}  eval_loss {el:.4f}  eval_acc {ea:.4f}")
    return h

In [6]:
history_baseline = train(model, train_loader)
with open("history_baseline.json", "w") as f:
    json.dump(history_baseline, f, indent=2)

 1  loss 0.6385  acc 0.6450  eval_loss 0.6010  eval_acc 0.6567


 2  loss 0.5441  acc 0.7175  eval_loss 0.6003  eval_acc 0.6517


 3  loss 0.4955  acc 0.7550  eval_loss 0.6405  eval_acc 0.6219


 4  loss 0.4330  acc 0.7913  eval_loss 0.6274  eval_acc 0.6915


 5  loss 0.4048  acc 0.8087  eval_loss 0.6293  eval_acc 0.6617


 6  loss 0.3953  acc 0.8075  eval_loss 0.6332  eval_acc 0.7114


 7  loss 0.3437  acc 0.8363  eval_loss 0.6895  eval_acc 0.6716


 8  loss 0.2806  acc 0.8850  eval_loss 0.6497  eval_acc 0.7214


 9  loss 0.2236  acc 0.9100  eval_loss 0.7114  eval_acc 0.7164


10  loss 0.1584  acc 0.9387  eval_loss 1.0212  eval_acc 0.6368


## Q3 - median train accuracy

In [7]:
round(statistics.median(history_baseline['train_accuracy']), 2)

0.81

**Q3: 0.81** (0.808125).

## Q4 - std of train loss

In [8]:
round(float(np.std(history_baseline['train_loss'], ddof=0)), 3)

0.139

**Q4: 0.139.** The loss keeps dropping, from 0.64 to 0.16, while eval loss goes up to 1.02 in the last epoch, so the baseline starts to overfit.

## Augmentation

Same model and optimizer, 10 more epochs. The loader generator uses seed 43 like
`reference_train.py`.

In [9]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(50, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1),
                                 interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])

train_dataset_aug = datasets.ImageFolder("data/train", transform=train_transform_augmented)
train_loader_aug = DataLoader(train_dataset_aug, batch_size=20, shuffle=True, num_workers=0,
                              generator=torch.Generator().manual_seed(SEED + 1))

history_augmented = train(model, train_loader_aug)
with open("history_augmented.json", "w") as f:
    json.dump(history_augmented, f, indent=2)

 1  loss 0.6758  acc 0.6050  eval_loss 0.6552  eval_acc 0.6716


 2  loss 0.6129  acc 0.6475  eval_loss 0.6608  eval_acc 0.6269


 3  loss 0.5899  acc 0.6800  eval_loss 0.5787  eval_acc 0.7114


 4  loss 0.5923  acc 0.6750  eval_loss 0.5679  eval_acc 0.7114


 5  loss 0.5604  acc 0.7137  eval_loss 0.5680  eval_acc 0.7214


 6  loss 0.5567  acc 0.7075  eval_loss 0.5510  eval_acc 0.7015


 7  loss 0.5433  acc 0.7000  eval_loss 0.5291  eval_acc 0.7413


 8  loss 0.5317  acc 0.7288  eval_loss 0.5458  eval_acc 0.7065


 9  loss 0.5041  acc 0.7562  eval_loss 0.5557  eval_acc 0.7214


10  loss 0.5398  acc 0.7050  eval_loss 0.5462  eval_acc 0.7313


## Q5 - mean eval loss with augmentation

In [10]:
round(statistics.mean(history_augmented['evaluation_loss']), 3)

0.576

**Q5: 0.576.**

## Q6 - mean eval accuracy, last 5 epochs

In [11]:
round(statistics.mean(history_augmented['evaluation_accuracy'][-5:]), 2)

0.72

**Q6: 0.72** (0.7204).

With augmentation the train accuracy falls back to about 0.70, but eval loss settles around 0.55
instead of climbing. Both history files match `reference_train.py` from the course repo run on
the same data (torch 2.9.0+cpu, torchvision 0.24.0+cpu, Python 3.11).